# Week 1-2 · Document와 text splitter 조립하기

## 시나리오
긴 휴가 규정 두 개를 검색 가능한 chunk로 나누고, 원문 식별자가 각 chunk에 유지되는지 확인합니다.

## 학습 목표
- LangChain `Document`를 직접 생성한다.
- `RecursiveCharacterTextSplitter`의 크기와 overlap을 설정한다.
- chunk text와 metadata를 함께 관찰한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · 작은 정책 문서 만들기

In [ ]:
# 실행 순서: 1단계 · 작은 정책 문서 만들기에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · 작은 정책 문서 만들기.
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

practice_documents = [
    Document(page_content="연차 휴가는 시작일 기준 3영업일 전에 신청합니다. 긴급한 경우에는 관리자에게 사유를 알립니다.", metadata={"document_id": "leave-policy"}),
    Document(page_content="병가는 진료 확인 자료를 첨부하며, 개인정보는 필요한 범위에서만 처리합니다.", metadata={"document_id": "sick-policy"}),
]
[(doc.page_content, doc.metadata) for doc in practice_documents]

### 2단계 · splitter 구성과 실행

In [ ]:
# 실행 순서: 2단계 · splitter 구성과 실행에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · splitter 구성과 실행.
practice_splitter = RecursiveCharacterTextSplitter(
    chunk_size=35,
    chunk_overlap=8,
    separators=[". ", " ", ""],
)
practice_chunks = practice_splitter.split_documents(practice_documents)
[(chunk.page_content, chunk.metadata) for chunk in practice_chunks]

### 3단계 · 중간 결과 검증

In [ ]:
# 실행 순서: 3단계 · 중간 결과 검증에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · 중간 결과 검증.
assert len(practice_chunks) > len(practice_documents)
assert {chunk.metadata["document_id"] for chunk in practice_chunks} == {"leave-policy", "sick-policy"}
assert all(chunk.page_content.strip() for chunk in practice_chunks)
{"source_count": len(practice_documents), "chunk_count": len(practice_chunks)}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
chunk가 원문 metadata를 잃으면 답변은 출처를 인용할 수 없습니다. 너무 작은 chunk는 문맥을 잘라낼 수 있습니다.

## 실제 app 연결
Week 1 app은 같은 `Document → split_documents` 흐름으로 정책 원문을 citation 가능한 검색 단위로 만듭니다. 실제 app에서는 여기에 안정적인 `chunk_id`를 더합니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 `03_embeddings_pgvector_retrieval.ipynb`에서는 생성한 chunk를 질문과 비교해 관련 근거만 선택합니다.